---
## Module 3 - Neural Networks

**Neural Networks** are the core building block of modern AI - loosely inspired by biological neurons.

| Concept | Description |
|---------|-------------|
| Perceptron | Single neuron: weighted sum + threshold |
| MLP | Multiple layers of perceptrons |
| Backpropagation | Gradient-based weight update algorithm |
| Activation Functions | Non-linearities: ReLU, Sigmoid, Tanh, Softmax |
| CNN | Shared weight filters for spatial data |
| RNN / LSTM | Handles sequential/temporal data |
| Self-Organizing Maps | Unsupervised competitive learning topology |


### 3.1 Activation Functions - Visualization & Comparison

In [ ]:
x = np.linspace(-5, 5, 400)
activations = {
    'Sigmoid':     (1 / (1 + np.exp(-x)),   '#E53935'),
    'Tanh':        (np.tanh(x),              '#1E88E5'),
    'ReLU':        (np.maximum(0, x),        '#43A047'),
    'Leaky ReLU':  (np.where(x>0, x, 0.1*x),'#FB8C00'),
    'ELU':         (np.where(x>0, x, np.exp(x)-1), '#8E24AA'),
    'Softplus':    (np.log(1+np.exp(x)),     '#00ACC1'),
}
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.ravel()
for ax, (name, (y_act, col)) in zip(axes, activations.items()):
    ax.plot(x, y_act, color=col, lw=2.5)
    ax.axhline(0, color='black', lw=0.8, linestyle='--')
    ax.axvline(0, color='black', lw=0.8, linestyle='--')
    ax.set_title(name, fontweight='bold', fontsize=12)
    ax.set_xlabel('x'); ax.set_ylabel('f(x)')
    ax.set_ylim(-2, 5); ax.grid(True, alpha=0.3)
    # Add derivative
    dy = np.gradient(y_act, x)
    ax.plot(x, dy, color=col, lw=1.5, linestyle=':', alpha=0.7, label="f'(x)")
    ax.legend(fontsize=8)
plt.suptitle('Activation Functions & Their Derivatives', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/03_activation_functions.png', dpi=130, bbox_inches='tight')
plt.show()

# Properties table
df_act = pd.DataFrame({
    'Function':   ['Sigmoid','Tanh','ReLU','Leaky ReLU','ELU'],
    'Range':      ['(0,1)','(-1,1)','[0,)','(-,)','(-1,)'],
    'Vanishing Grad Risk': ['High','Medium','None (positive)','Low','Low'],
    'Typical Use': ['Output (binary)','Hidden','Hidden (default)','Hidden','Hidden'],
    'Computationally': ['Moderate','Moderate','Fast','Fast','Moderate'],
})
print(df_act.to_string(index=False))

### 3.2 Perceptron - From Scratch (AND / OR Gate)

In [ ]:
class Perceptron:
    def __init__(self, lr=0.1, epochs=100):
        self.lr, self.epochs = lr, epochs
    def fit(self, X, y):
        self.w = np.zeros(X.shape[1])
        self.b = 0
        self.errors_ = []
        for _ in range(self.epochs):
            err = 0
            for xi, yi in zip(X, y):
                pred = self.predict(xi)
                delta = self.lr * (yi - pred)
                self.w += delta * xi
                self.b += delta
                err   += int(delta != 0)
            self.errors_.append(err)
    def predict(self, X):
        return np.where(np.dot(X, self.w) + self.b >= 0.5, 1, 0)

# Logic gates
gates = {
    'AND': (np.array([[0,0],[0,1],[1,0],[1,1]]), np.array([0,0,0,1])),
    'OR':  (np.array([[0,0],[0,1],[1,0],[1,1]]), np.array([0,1,1,1])),
}
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (gate, (Xg, yg)) in zip(axes, gates.items()):
    p = Perceptron(lr=0.1, epochs=30)
    p.fit(Xg, yg)
    preds = p.predict(Xg)
    correct = (preds == yg).all()
    colors = ['#E53935' if l==0 else '#43A047' for l in yg]
    ax.scatter(Xg[:,0], Xg[:,1], c=colors, s=300, zorder=3, edgecolors='black', lw=2)
    for i,(xi,yi_l,pred) in enumerate(zip(Xg,yg,preds)):
        ax.text(xi[0]+0.04, xi[1]+0.04, f'{yi_l}{pred}', fontsize=10)
    if p.w[1] != 0:
        xline = np.linspace(-0.5,1.5,100)
        yline = (-p.w[0]*xline - p.b + 0.5) / p.w[1]
        ax.plot(xline, yline, 'b--', lw=2, label='Decision boundary')
    ax.set_xlim(-0.5,1.5); ax.set_ylim(-0.5,1.5)
    ax.set_title(f'{gate} Gate - {" Learned" if correct else " Not separable"}', fontweight='bold')
    ax.set_xlabel('Input A'); ax.set_ylabel('Input B'); ax.legend()
plt.suptitle('Perceptron - Logic Gate Learning', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/03_perceptron.png', dpi=130, bbox_inches='tight')
plt.show()
print("AND predictions:", p.predict(gates['AND'][0]))

### 3.3 Multi-Layer Perceptron (MLP) - MNIST Handwritten Digits

In [ ]:
#  Load MNIST 
(X_tr, y_tr), (X_te, y_te) = mnist.load_data()
X_tr = X_tr.reshape(-1, 784).astype('float32') / 255.0
X_te = X_te.reshape(-1, 784).astype('float32') / 255.0
Y_tr = to_categorical(y_tr, 10); Y_te = to_categorical(y_te, 10)

#  Build MLP 
mlp = keras.Sequential([
    layers.Dense(256, activation='relu',    input_shape=(784,), name='hidden1'),
    layers.Dropout(0.3, name='dropout1'),
    layers.Dense(128, activation='relu',    name='hidden2'),
    layers.Dropout(0.2, name='dropout2'),
    layers.Dense(64,  activation='relu',    name='hidden3'),
    layers.Dense(10,  activation='softmax', name='output'),
], name='MLP_MNIST')
mlp.summary()

mlp.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
early_stop = callbacks.EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True)
history_mlp = mlp.fit(X_tr, Y_tr, epochs=25, batch_size=256,
                       validation_split=0.1, callbacks=[early_stop], verbose=1)

#  Evaluate 
_, test_acc = mlp.evaluate(X_te, Y_te, verbose=0)
print(f"\n MLP Test Accuracy: {test_acc*100:.2f}%")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(history_mlp.history['accuracy'],     label='Train Acc', lw=2)
axes[0].plot(history_mlp.history['val_accuracy'], label='Val Acc',   lw=2)
axes[0].set_title('MLP - Accuracy Curves', fontweight='bold'); axes[0].legend()
axes[1].plot(history_mlp.history['loss'],     label='Train Loss', lw=2)
axes[1].plot(history_mlp.history['val_loss'], label='Val Loss',   lw=2)
axes[1].set_title('MLP - Loss Curves', fontweight='bold'); axes[1].legend()
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/03_mlp_mnist.png', dpi=130, bbox_inches='tight'); plt.show()

#  Confusion matrix 
y_pred = np.argmax(mlp.predict(X_te, verbose=0), axis=1)
cm = confusion_matrix(y_te, y_pred)
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=range(10), yticklabels=range(10))
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title(f'MLP Confusion Matrix - Test Acc: {test_acc*100:.2f}%', fontweight='bold')
plt.tight_layout(); plt.savefig(f'{SAVE_DIR}/03_mlp_confusion.png', dpi=130, bbox_inches='tight'); plt.show()

### 3.x Backpropagation  Manual Forward & Backward Pass Visualization
Backpropagation computes gradients layer-by-layer using the chain rule,
then updates weights via gradient descent. This cell builds a 2-layer
network from scratch  no Keras  so every gradient is visible.

In [ ]:
#  Backpropagation from scratch on XOR problem 
def sigmoid(z):      return 1 / (1 + np.exp(-z))
def sigmoid_d(z):    return sigmoid(z) * (1 - sigmoid(z))
def mse(y, yhat):    return np.mean((y - yhat)**2)

np.random.seed(42)
X_xor = np.array([[0,0],[0,1],[1,0],[1,1]], dtype=float)
y_xor = np.array([[0],[1],[1],[0]],         dtype=float)

# Weights: input(2)  hidden(4)  output(1)
W1 = np.random.randn(2, 4) * 0.5
b1 = np.zeros((1, 4))
W2 = np.random.randn(4, 1) * 0.5
b2 = np.zeros((1, 1))

lr = 0.5
losses, W1_norms, W2_norms = [], [], []

for epoch in range(5000):
    #  Forward pass 
    Z1   = X_xor @ W1 + b1          # (4,4)
    A1   = sigmoid(Z1)              # hidden activations
    Z2   = A1  @ W2 + b2            # (4,1)
    A2   = sigmoid(Z2)              # output
    loss = mse(y_xor, A2)

    #  Backward pass (chain rule) 
    dL_A2 = -2 * (y_xor - A2) / len(y_xor)
    dA2   = sigmoid_d(Z2)
    dZ2   = dL_A2 * dA2

    dW2   = A1.T @ dZ2
    db2   = dZ2.sum(axis=0, keepdims=True)

    dA1   = dZ2 @ W2.T
    dZ1   = dA1 * sigmoid_d(Z1)
    dW1   = X_xor.T @ dZ1
    db1   = dZ1.sum(axis=0, keepdims=True)

    #  Weight update 
    W2 -= lr * dW2;  b2 -= lr * db2
    W1 -= lr * dW1;  b1 -= lr * db1

    losses.append(loss)
    W1_norms.append(np.linalg.norm(dW1))
    W2_norms.append(np.linalg.norm(dW2))

print("Final predictions vs truth:")
preds = (A2 > 0.5).astype(int)
df_bp = pd.DataFrame({'Input': ['[0,0]','[0,1]','[1,0]','[1,1]'],
                       'True':  y_xor.ravel().astype(int),
                       'Pred':  preds.ravel(),
                       'Raw Output': A2.ravel().round(4)})
print(df_bp.to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(losses, color='#E53935', lw=2)
axes[0].set_title('Loss over Epochs', fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('MSE Loss')
axes[0].set_yscale('log')

axes[1].plot(W1_norms, label='||dW1|| Layer 1', color='#1E88E5', lw=2)
axes[1].plot(W2_norms, label='||dW2|| Layer 2', color='#43A047', lw=2)
axes[1].set_title('Gradient Norms per Layer', fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Gradient Norm')
axes[1].legend(); axes[1].set_yscale('log')

# Weight heatmap final W1
sns.heatmap(W1, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            ax=axes[2], xticklabels=[f'H{i}' for i in range(4)],
            yticklabels=['x1','x2'])
axes[2].set_title('Learned W1 Weights (inputhidden)', fontweight='bold')

plt.suptitle('Backpropagation  Manual Gradient Descent on XOR', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/03x_backpropagation.png', dpi=130, bbox_inches='tight')
plt.show()

### 3.4 Convolutional Neural Network (CNN) - MNIST

In [ ]:
(X_tr, y_tr), (X_te, y_te) = mnist.load_data()
X_tr_c = X_tr.reshape(-1,28,28,1).astype('float32')/255.0
X_te_c = X_te.reshape(-1,28,28,1).astype('float32')/255.0
Y_tr_c = to_categorical(y_tr, 10); Y_te_c = to_categorical(y_te, 10)

cnn = keras.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', padding='same', input_shape=(28,28,1)),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),
    layers.Conv2D(64, (3,3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),
    layers.Conv2D(64, (3,3), activation='relu', padding='same'),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(10, activation='softmax'),
], name='CNN_MNIST')
cnn.summary()

cnn.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
early_stop2 = callbacks.EarlyStopping(monitor='val_accuracy', patience=4, restore_best_weights=True)
history_cnn = cnn.fit(X_tr_c, Y_tr_c, epochs=20, batch_size=256,
                       validation_split=0.1, callbacks=[early_stop2], verbose=1)

_, cnn_acc = cnn.evaluate(X_te_c, Y_te_c, verbose=0)
print(f"\n CNN Test Accuracy: {cnn_acc*100:.2f}%")

#  Visualize filters from first conv layer 
filters = cnn.layers[0].get_weights()[0]
fig, axes = plt.subplots(4, 8, figsize=(14, 7))
for i, ax in enumerate(axes.ravel()):
    if i < filters.shape[3]:
        ax.imshow(filters[:,:,0,i], cmap='viridis')
    ax.axis('off')
plt.suptitle(f'CNN Layer 1 - 32 Learned Filters | Test Acc: {cnn_acc*100:.2f}%',
             fontsize=13, fontweight='bold')
plt.tight_layout(); plt.savefig(f'{SAVE_DIR}/03_cnn_filters.png', dpi=130, bbox_inches='tight'); plt.show()

print(f"\nMLP vs CNN:\n  MLP: {history_mlp.history['val_accuracy'][-1]*100:.2f}%\n  CNN: {cnn_acc*100:.2f}%")

### 3.5 LSTM & RNN - Time Series Prediction (Sine Wave)

In [ ]:
#  Generate sine wave data 
t = np.linspace(0, 8*np.pi, 1000)
signal = np.sin(t) + 0.1*np.random.randn(len(t))

def make_sequences(data, seq_len=50):
    X_s, y_s = [], []
    for i in range(len(data)-seq_len):
        X_s.append(data[i:i+seq_len]); y_s.append(data[i+seq_len])
    return np.array(X_s)[..., None], np.array(y_s)

SEQ_LEN = 50
X_seq, y_seq = make_sequences(signal, SEQ_LEN)
split = int(0.8*len(X_seq))
X_str, X_ste = X_seq[:split], X_seq[split:]
y_str, y_ste = y_seq[:split], y_seq[split:]

models_seq = {}
for arch, lyr in [('SimpleRNN', layers.SimpleRNN(64)),
                  ('LSTM',      layers.LSTM(64)),
                  ('GRU',       layers.GRU(64))]:
    m = keras.Sequential([
        layers.Input(shape=(SEQ_LEN,1)),
        lyr,
        layers.Dense(32, activation='relu'),
        layers.Dense(1)
    ], name=arch)
    m.compile(optimizer='adam', loss='mse')
    m.fit(X_str, y_str, epochs=10, batch_size=64, validation_split=0.1, verbose=0)
    models_seq[arch] = m

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
t_test = t[split+SEQ_LEN:]
for ax, (name, m) in zip(axes, models_seq.items()):
    preds = m.predict(X_ste, verbose=0).ravel()
    mse_v = mean_squared_error(y_ste, preds)
    ax.plot(t_test[:len(y_ste)], y_ste,  'b-',  lw=1.5, alpha=0.7, label='True Signal')
    ax.plot(t_test[:len(preds)], preds, 'r--', lw=1.5, label=f'{name} Prediction')
    ax.set_title(f'{name}  |  MSE = {mse_v:.5f}', fontweight='bold')
    ax.legend(loc='upper right'); ax.set_ylabel('Value')
axes[-1].set_xlabel('Time')
plt.suptitle('RNN vs LSTM vs GRU - Sine Wave Forecasting', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.savefig(f'{SAVE_DIR}/03_lstm_rnn.png', dpi=130, bbox_inches='tight'); plt.show()

df_seq = pd.DataFrame([{'Model':k, 'Test MSE':f'{mean_squared_error(y_ste, v.predict(X_ste, verbose=0).ravel()):.5f}'}
                        for k,v in models_seq.items()])
print(df_seq.to_string(index=False))

### 3.6 Self-Organizing Maps (SOM)

In [ ]:
from minisom import MiniSom
iris = load_iris()
X_som = StandardScaler().fit_transform(iris.data)

som = MiniSom(10, 10, 4, sigma=1.5, learning_rate=0.5, random_seed=42)
som.random_weights_init(X_som)
som.train_random(X_som, 5000, verbose=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
# Distance map (U-matrix)
im = axes[0].pcolor(som.distance_map().T, cmap='bone_r')
plt.colorbar(im, ax=axes[0])
markers = ['o', 's', '^']
colors  = ['#E53935','#43A047','#1E88E5']
for x_row, t in zip(X_som, iris.target):
    w = som.winner(x_row)
    axes[0].plot(w[0]+.5, w[1]+.5, markers[t], markerfacecolor='None',
                 markeredgecolor=colors[t], markersize=8, markeredgewidth=2)
axes[0].set_title('SOM U-Matrix (darker = more distinct boundary)', fontweight='bold')
patches = [mpatches.Patch(color=colors[i], label=iris.target_names[i]) for i in range(3)]
axes[0].legend(handles=patches, loc='upper right')

# Activation frequency
axes[1].pcolor(som.activation_response(X_som).T, cmap='YlOrRd')
axes[1].set_title('SOM Activation Frequency Map', fontweight='bold')
plt.suptitle('Self-Organizing Map - Iris Dataset', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.savefig(f'{SAVE_DIR}/03_som.png', dpi=130, bbox_inches='tight'); plt.show()